In [ ]:
#!/usr/bin/env python3
"""
src/sat_solver.py
Propositional SAT Constraint Engine (V_SAT) for Feature Model Validity Enforcement.
Computes m_SAT in {0, 1}^{n+1} for action masking in the Constrained MDP.
"""

import numpy as np

try:
    from pysat.solvers import Glucose3
    HAS_PYSAT = True
except ImportError:
    HAS_PYSAT = False


class FeatureModelSATSolver:
    """Enforces propositional feature-model constraints Phi on candidate configurations."""

    def __init__(self, n_features: int, clauses: list[list[int]]):
        self.n_features = n_features
        self.clauses = clauses

    def is_valid(self, config: np.ndarray) -> bool:
        """Check whether binary configuration vector c satisfies all clauses in Phi."""
        if HAS_PYSAT:
            with Glucose3(bootstrap_with=self.clauses) as solver:
                assumptions = [
                    (i + 1) if config[i] == 1 else -(i + 1)
                    for i in range(self.n_features)
                ]
                return bool(solver.solve(assumptions=assumptions))
        # Exact fallback evaluator
        for clause in self.clauses:
            satisfied = False
            for lit in clause:
                idx = abs(lit) - 1
                if (lit > 0 and config[idx] == 1) or (lit < 0 and config[idx] == 0):
                    satisfied = True
                    break
            if not satisfied:
                return False
        return True

    def compute_action_mask(self, config: np.ndarray, candidate_indices: np.ndarray | None = None) -> np.ndarray:
        """
        Compute binary validity mask m_SAT of length (n_features + 1).
        Index 0 = retain current configuration (always valid if current config is valid).
        Index i (1..n) = toggle feature (i-1).
        """
        mask = np.zeros(self.n_features + 1, dtype=np.float32)
        mask[0] = 1.0  # No-op action
        indices = candidate_indices if candidate_indices is not None else np.arange(self.n_features)
        for idx in indices:
            trial = config.copy()
            trial[idx] = 1 - trial[idx]
            if self.is_valid(trial):
                mask[idx + 1] = 1.0
        return mask